In [15]:
import pandas as pd
from datetime import datetime, timezone

In [16]:
rawdata=pd.read_csv('precio.csv')
rawdata2=pd.read_csv('criptomoneda.csv')

In [17]:
df_precios=rawdata
df_criptos=rawdata2

In [18]:
df_precios

,rank,price_usd,percent_change_24h,percent_change_7d,price_btc,date,id
0,1,123856.990000,-0.91,12.46,1.000000,2025-10-06T06:36:19.455475,90
1,2,4561.130000,-0.12,12.96,0.036832,2025-10-06T06:36:19.455475,80
2,3,2.990000,-1.55,6.02,0.000024,2025-10-06T06:36:19.455475,58
3,4,1.000000,0.01,-0.11,0.000008,2025-10-06T06:36:19.455475,518
4,5,1214.800000,3.52,24.12,0.009810,2025-10-06T06:36:19.455475,2710
...,...,...,...,...,...,...,...
1195,96,2.940000,0.40,11.08,0.000024,2025-10-07T07:16:54.894993,47371
1196,97,0.785620,1.86,8.88,0.000006,2025-10-07T07:16:54.894993,121613
1197,98,0.188813,-0.51,15.80,0.000002,2025-10-07T07:16:54.894993,447
1198,99,0.148487,1.22,3.47,0.000001,2025-10-07T07:16:54.894993,121599


In [19]:
df_total = pd.merge(df_criptos, df_precios, left_on='id', right_on='id', how='inner')

In [20]:
df_total

,id,name,symbol,rank,price_usd,percent_change_24h,percent_change_7d,price_btc,date
0,90,Bitcoin,BTC,1,123856.990000,-0.91,12.46,1.000000,2025-10-06T06:36:19.455475
1,90,Bitcoin,BTC,1,123990.620000,-0.75,12.54,1.000000,2025-10-06T06:41:07.322076
2,90,Bitcoin,BTC,1,123896.360000,-0.88,12.35,1.000000,2025-10-06T07:02:55.248619
3,90,Bitcoin,BTC,1,123896.360000,-0.88,12.35,1.000000,2025-10-06T07:04:14.966707
4,90,Bitcoin,BTC,1,123871.130000,-0.93,12.36,1.000000,2025-10-06T07:13:26.298423
...,...,...,...,...,...,...,...,...,...
1180,121613,dogwifhat,WIF,99,0.764329,-4.97,3.97,0.000006,2025-10-06T08:28:35.677450
1181,121613,dogwifhat,WIF,99,0.768207,-4.14,4.63,0.000006,2025-10-06T09:07:24.027905
1182,121613,dogwifhat,WIF,97,0.778633,-0.43,4.46,0.000006,2025-10-06T11:34:51.959044
1183,121613,dogwifhat,WIF,98,0.775473,-0.79,3.83,0.000006,2025-10-06T11:48:40.014574


In [21]:
df_bitcoin = df_total[df_total['name'] == "Bitcoin"].copy()
df_bitcoin['date'] = pd.to_datetime(df_bitcoin['date'], utc=True)



In [22]:
df_bitcoin

,id,name,symbol,rank,price_usd,percent_change_24h,percent_change_7d,price_btc,date
0,90,Bitcoin,BTC,1,123856.99,-0.91,12.46,1.0,2025-10-06 06:36:19.455475+00:00
1,90,Bitcoin,BTC,1,123990.62,-0.75,12.54,1.0,2025-10-06 06:41:07.322076+00:00
2,90,Bitcoin,BTC,1,123896.36,-0.88,12.35,1.0,2025-10-06 07:02:55.248619+00:00
3,90,Bitcoin,BTC,1,123896.36,-0.88,12.35,1.0,2025-10-06 07:04:14.966707+00:00
4,90,Bitcoin,BTC,1,123871.13,-0.93,12.36,1.0,2025-10-06 07:13:26.298423+00:00
5,90,Bitcoin,BTC,1,123937.85,-0.86,12.53,1.0,2025-10-06 07:18:51.038274+00:00
6,90,Bitcoin,BTC,1,123829.22,-0.69,12.25,1.0,2025-10-06 07:39:54.729602+00:00
7,90,Bitcoin,BTC,1,123608.58,-0.86,11.81,1.0,2025-10-06 08:28:35.677450+00:00
8,90,Bitcoin,BTC,1,123847.39,-0.61,11.76,1.0,2025-10-06 09:07:24.027905+00:00
9,90,Bitcoin,BTC,1,124363.22,0.83,10.95,1.0,2025-10-06 11:34:51.959044+00:00


In [23]:
from influxdb_client import InfluxDBClient

In [24]:
my_token = "LbKYCkDRS7jmz-k_PBa_Zq5doOV2XFkgr3pWLBMdHeVwxw7agEZtXjc1dGkTsc3gVl9r6ihU9exA3sw0kJexfg=="

In [25]:
client = InfluxDBClient(url="http://localhost:8086", token=my_token, org="somorrostro")

In [26]:
write_api = client.write_api()

In [27]:
write_api.write(
    bucket="pruebas-bucket",
    org="somorrostro",
    record=df_bitcoin,
    data_frame_measurement_name="Bitcoin",
    data_frame_tag_columns=['name','symbol'],
    data_frame_field_columns=['rank','price_usd','percent_change_24h','percent_change_7d','price_btc'],
    data_frame_time_column='date',
    write_precision='ns'  # también probar 'ms' o 's'
)


In [36]:
import pandas as pd
from influxdb_client import InfluxDBClient
from influxdb_client.client.write_api import SYNCHRONOUS

# --- Cargar CSVs ---
rawdata = pd.read_csv('precio.csv')
rawdata2 = pd.read_csv('criptomoneda.csv')

df_precios = rawdata
df_criptos = rawdata2

# --- Merge de dataframes ---
df_total = pd.merge(df_criptos, df_precios, left_on='id', right_on='id', how='inner')

# --- Filtrar Bitcoin y hacer copia ---
df_bitcoin = df_total[df_total['name'] == "Bitcoin"].copy()

# --- Convertir timestamp a datetime UTC ---
df_bitcoin['date'] = pd.to_datetime(df_bitcoin['date'], utc=True)

# --- Definir tags y fields ---
tag_columns = ['name', 'symbol']
field_columns = ['rank', 'price_usd', 'percent_change_24h', 'percent_change_7d', 'price_btc']

# --- Limpiar DataFrame de posibles NaNs ---
df_bitcoin_clean = df_bitcoin.dropna(subset=field_columns + tag_columns)

# --- Conectar a InfluxDB ---
my_token = "LbKYCkDRS7jmz-k_PBa_Zq5doOV2XFkgr3pWLBMdHeVwxw7agEZtXjc1dGkTsc3gVl9r6ihU9exA3sw0kJexfg=="
client = InfluxDBClient(url="http://localhost:8086", token=my_token, org="somorrostro")

# --- Crear write_api en modo SYNCHRONOUS ---
write_api = client.write_api(write_options=SYNCHRONOUS)

# --- Escribir en InfluxDB ---
write_api.write(
    bucket="hola",
    org="somorrostro",
    record=df_bitcoin_clean,
    data_frame_measurement_name="Bitcoin",
    data_frame_tag_columns=tag_columns,
    data_frame_field_columns=field_columns,
    data_frame_time_column='date',
    write_precision='ns'
)

print("Datos de Bitcoin enviados a InfluxDB correctamente.")


Datos de Bitcoin enviados a InfluxDB correctamente.


In [38]:
from influxdb_client import InfluxDBClient

client = InfluxDBClient(url="http://localhost:8086", token=my_token, org="somorrostro")
bucket = client.buckets_api().find_bucket_by_name("hola")
if bucket:
    print("Bucket encontrado ✅")
else:
    print("Bucket NO encontrado ❌")


Bucket encontrado ✅


In [37]:
import pandas as pd

test_df = pd.DataFrame({
    'date': [pd.Timestamp('2025-10-06 06:36:19', tz='UTC')],
    'name': ['Bitcoin'],
    'symbol': ['BTC'],
    'rank': [1],
    'price_usd': [123456.78],
    'percent_change_24h': [-0.91],
    'percent_change_7d': [12.46],
    'price_btc': [1.0]
})

write_api.write(
    bucket="hola",
    org="somorrostro",
    record=test_df,
    data_frame_measurement_name="Bitcoin",
    data_frame_tag_columns=['name', 'symbol'],
    data_frame_field_columns=['rank', 'price_usd', 'percent_change_24h', 'percent_change_7d', 'price_btc'],
    data_frame_time_column='date'
)
print("Fila de prueba escrita ✅")


Fila de prueba escrita ✅
